# Agentic Librarian Analysis

markdown
intro-1
markdown
# Agentic Librarian Analysis

This notebook explains whether the Student + Librarian demo behaves like an agent, which parts are already agentic, the gaps that prevent safe autonomy, and a concrete, prioritized roadmap to increase agentic capability. It finishes with a guarded LLM evaluation harness you can run locally.
markdown
what-is-agentic-2
markdown
## What we mean by 'agentic'

An agentic system (practical definition for this project) demonstrates:
- Autonomy: initiates actions without immediate human instruction when appropriate.
- Goal-directed behavior: represents goals and decomposes them into steps.
- Stateful observation: records environment state and uses it for decisions.
- Effectful actions: updates systems, triggers notifications, or modifies data.
- Safe oversight: includes policies, human-in-the-loop controls, and rollback mechanisms.
markdown
summary-3
markdown
## Summary: is the librarian app agentic?

Short answer: partially. The demo contains agent-like modules (recommendation, LLM blurbs, stateful in-memory lifecycle) but it's primarily reactive and lacks the scaffolding for safe autonomous operation in production. The sections below map components to agentic properties, list concrete gaps, and provide implementable recommendations and a rollout roadmap.
markdown
markdown
intro-1
markdown
# Agentic Librarian Analysis

This notebook explains whether the Student + Librarian demo behaves like an agent, which parts are already agentic, the gaps that prevent safe autonomy, and a concrete, prioritized roadmap to increase agentic capability. It finishes with a guarded LLM evaluation harness you can run locally.
markdown
what-is-agentic-2
markdown
## What we mean by 'agentic'

An agentic system (practical definition for this project) demonstrates:
- Autonomy: initiates actions without immediate human instruction when appropriate.
- Goal-directed behavior: represents goals and decomposes them into steps.
- Stateful observation: records environment state and uses it for decisions.
- Effectful actions: updates systems, triggers notifications, or modifies data.
- Safe oversight: includes policies, human-in-the-loop controls, and rollback mechanisms.
markdown
summary-3
markdown
## Summary: is the librarian app agentic?

Short answer: partially. The demo contains agent-like modules (recommendation, LLM blurbs, stateful in-memory lifecycle) but it's primarily reactive and lacks the scaffolding for safe autonomous operation in production. The sections below map components to agentic properties, list concrete gaps, and provide implementable recommendations and a rollout roadmap.
markdown
mapping-4
markdown
## Component mapping to agentic properties

- Recommendation engine: goal-directed logic exists (match profiles → books), but it's synchronous and only runs on request (reactive). To be agentic it must schedule recommendations, re-evaluate goals, and take actions (e.g., send nudges).
- LLM blurb generator: performs effectful actions (creates content) but lacks safety checks, provenance, and a reject/approve workflow.
- In-memory lifecycle & queues: provides transient state but no durable memory, no audit trail, and no crash recovery.
- Gradio UI: good human-in-the-loop surface; not a control plane for autonomous behaviors.
- Metrics & export scaffolding: present but minimal; needs richer observability (decision traces, feedback loops).
markdown
gaps-5
markdown
## Key gaps preventing production-grade agentic behavior

1) Durable state & auditability: move from in-memory dicts to a persistent store (SQL/NoSQL/event store) with append-only audit logs and versioning for user profiles and decisions.

2) Background execution & scheduling: no worker or scheduler exists to run autonomous tasks (daily digest, periodic re-ranking, automatic nudges).

3) Explicit goals & planners: the system needs a goal representation (user retention, reading progression) and a planner to sequence actions toward goals.

4) Safety, policy and human oversight: LLM outputs require filters, human approval flows for high-impact actions, and an enforceable policy engine (e.g., OPA).

5) Observability & evaluation: decision traces, outcome metrics, A/B evaluations, and continuous model evaluation to detect drift.
markdown
recommendations-6
markdown
## Concrete recommendations to increase agentic capability (prioritized)

Below are actionable changes ordered by near-term (safe, low-risk) → medium-term (infrastructure) → long-term (autonomy & planning).

### Near-term (0–2 weeks) — low risk, high value
- Persist critical state: move librarian queues, user profiles, and book lifecycle events into a lightweight SQLite/Postgres DB. Capture timestamps and actor (agent vs human) for every state change.
- Add a task runner fallback: integrate APScheduler or RQ for simple scheduled jobs. This enables periodic re-ranking or nightly digest jobs without full infra.
- Add a policy filter around LLM outputs: simple regex checks + a configurable allow/deny list and profanity/PII detectors to avoid obvious violations.
- Extend logging: log decision inputs, outputs, model metadata, and a request_id to correlate traces.

### Medium-term (2–8 weeks) — infrastructure & safety
- Introduce a worker queue (Redis + RQ / Celery, or Temporal for stronger ordering & retries). Use it for autonomous actions and for executing LLM calls outside request threads.
- Implement a Goal Store & Planner prototype: start with rule-based goals and a simple planner that selects candidate actions (send reminder, recommend list, escalate to librarian).
- Human-in-the-loop approval pipeline: for higher-risk LLM outputs, route to librarians for explicit approval via the UI before actioning (e.g., sending a student-facing message).
- Integrate a policy engine: OPA or a small custom policy layer to encode constraints (no personal data, no off-limits content).

### Long-term (8+ weeks) — autonomy with verification
- Replace rule planner with a scoped planner/search approach or integrate a planning library (PDDL-style or a simple MCTS over discrete actions).
- Build canary/autonomy levels: expose feature flags that enable different degrees of autonomy per cohort (e.g., pilot group gets automated nudges).
- Continuous evaluation loop: automated human preference collection, offline backtests, and model-based factuality checks that feed into training or tuning.
- Formalize rollback & safety: automated rollback triggers based on KPI regression, and an operator dashboard for fast intervention.
markdown
roadmap-7
markdown
## Implementation roadmap (example 90-day plan)

- Week 1–2: Add persistence (SQLite/Postgres) + migrate in-memory state, add request_id logging.
- Week 3–4: Add APScheduler / RQ to run nightly recommendation jobs and simple retry/backoff.
- Week 5–8: Prototype Goal Store & rule-based planner; build LLM safety filter and human approval UI flow.
- Week 9–12: Move to queue-backed workers (Redis/Celery or Temporal), run small pilot cohort with autonomy level 1 (automated nudges only).
- Ongoing: build LLM evaluation pipeline, A/B testing for autonomy levels, and monitoring/rollback controls.
markdown
llm-eval-intro-8
markdown
## LLM Evaluation — goals and metrics

Evaluation should cover three families of metrics: relevance (does the blurb match intent?), safety (PII, toxicity, policy violations), and utility (did it change behavior?). Suggested metrics:
- Relevance: overlap, embedding similarity, or human preference labels.
- Safety: binary triggers for PII/profanity and a severity score from model-based detectors.
- Utility: click-through, acceptance by librarians, change in reading behavior over time.
code
llm-eval-guarded-9
python
# Guarded LLM eval harness (expanded)
# This cell is defensive: optional deps are loaded only when available.
from typing import List, Dict
import json
import re

HAS_EMBED = False
try:
    from sentence_transformers import SentenceTransformer, util
    HAS_EMBED = True
except Exception:
    # embeddings unavailable; we'll fall back to overlap heuristics
    pass

def simple_relevance_score(candidate: str, query: str) -> float:
    # token overlap normalized by query length
    s_c = set(re.findall(r'\w+', candidate.lower()))
    s_q = set(re.findall(r'\w+', query.lower()))
    if not s_q:
        return 0.0
    return len(s_c & s_q) / len(s_q)

def embed_and_score(candidates: List[str], query: str) -> Dict:
    if HAS_EMBED:
        model = SentenceTransformer('all-MiniLM-L6-v2')
        q_emb = model.encode(query, convert_to_tensor=True)
        c_emb = model.encode(candidates, convert_to_tensor=True)
        scores = util.cos_sim(q_emb, c_emb).cpu().numpy().tolist()[0]
        return {'method': 'embed', 'scores': scores}
    else:
        return {'method': 'overlap', 'scores': [simple_relevance_score(c, query) for c in candidates]}

def contains_personal_data(text: str) -> bool:
    # naive detectors; replace with better PII detection in prod
    patterns = [r'\b\d{3}[-.]?\d{2}[-.]?\d{4}\b', r'\b\d{10}\b']
    for p in patterns:
        if re.search(p, text):
            return True
    return False

# Simple planner stub (rule-based)
def plan_for_goal(goal: str, context: Dict) -> List[str]:
    # Returns a list of action names (very small example)
    actions = []
    if 'increase_reading' in goal:
        actions = ['recommend_books', 'send_nudge', 'schedule_followup']
    else:
        actions = ['recommend_books']
    return actions

print('Guarded eval harness ready. HAS_EMBED=', HAS_EMBED)
code
eval-examples-10
python
# Small automated example evaluations
candidates = ["A Tale of Two Cities", "The Cat in the Hat", "Advanced Python Programming"]
query = "beginner Python book"
scores = embed_and_score(candidates, query)
print('Scoring method:', scores.get('method'))
for c, s in zip(candidates, scores['scores']):
    print(f'{s:.3f}', c)

# Example blurb eval
def evaluate_blurb(blurb: str, prompt: str) -> Dict:
    rel = simple_relevance_score(blurb, prompt)
    pii = contains_personal_data(blurb)
    fluency = max(0.0, 1.0 - (sum(1 for ch in blurb if ord(ch) < 32) / max(1, len(blurb))))
    return {'relevance': rel, 'pii': pii, 'fluency': fluency}

print(evaluate_blurb('This beginner friendly Python book teaches lists and loops.', query))
markdown
closing-11
markdown
## Next steps & further reading

- Formal LLM evals: human preference labeling, factuality checks, and offline backtests.
- Safety audits: red-team prompts, content filters, and continuous monitoring.
- Agentic verification: plan pilots with graduated autonomy levels and clearly defined rollback triggers.
markdown
glossary-12
markdown
## Glossary: short definitions for acronyms and terms used

- Regex: short for Regular Expression, a compact language for pattern-matching text (useful for simple PII or profanity checks).
- RQ: Redis Queue, a simple Python library for background jobs that uses Redis as a broker. Good for lightweight task queues and scheduled work.
- OPA: Open Policy Agent, a policy engine that lets you express, enforce, and test fine-grained access and behavior policies separately from application code.
- PDDL: Planning Domain Definition Language, a classical AI planning language used to describe actions, predicates, and goals for automated planners.
- MCTS: Monte Carlo Tree Search, a search algorithm that samples possible action sequences (rollouts) to estimate the value of actions; commonly used in planning under uncertainty.